# Studi Komparatif Generative Models — Pipeline Colab
**Deep Learning (D6421) · Sesi 07** — Vanilla AE, Sparse AE, VAE, Vanilla GAN, DCGAN, DDPM + pembuktian batasan MLP Classifier.

Notebook ini hanya **pengendali**: semua logika ada di `src/*.py` pada repositori GitHub.
Jalankan sel dari atas ke bawah. **Pastikan Runtime → Change runtime type → GPU (T4).**

Urutan: setup → (opsional) smoke test → training → eksperimen pembuktian MLP → evaluasi → figur.

## 1. Setup

In [ ]:
# ====== EDIT BAGIAN INI ======
GITHUB_URL = "https://github.com/USERNAME/NAMA-REPO.git"   # <- ganti dengan URL repo publik Anda
REPO_DIR   = "/content/NAMA-REPO"                           # <- samakan dengan nama repo
# =============================
import os
print("GPU tersedia:", __import__("torch").cuda.is_available())
!nvidia-smi -L

In [ ]:
# Simpan semua output (checkpoint, hasil, figur) ke Google Drive agar tidak hilang saat sesi Colab putus
from google.colab import drive
drive.mount("/content/drive")
os.environ["DL_OUT_DIR"]  = "/content/drive/MyDrive/dl_sesi07/outputs"
os.environ["DL_DATA_DIR"] = "/content/data"
os.makedirs(os.environ["DL_OUT_DIR"], exist_ok=True)
print("Output ->", os.environ["DL_OUT_DIR"])

In [ ]:
# Clone repositori & pasang dependensi
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull
!pip install -q -r requirements.txt

## 2. (Opsional) Smoke test
Mengecek seluruh kode berjalan tanpa error memakai data kecil dan 1 epoch (~2-3 menit).
Memakai folder output terpisah, sehingga **tidak** mengganggu hasil asli.

In [ ]:
!DL_SMOKE=1 DL_OUT_DIR=/content/smoke_outputs python -m src.train --model all --force
!DL_SMOKE=1 DL_OUT_DIR=/content/smoke_outputs python -m src.mlp_proof

## 3. Training
Checkpoint yang sudah ada **dilewati otomatis**, sehingga jika sesi putus Anda cukup menjalankan ulang sel ini.
Ubah jumlah epoch di `src/config.py` (`_EPOCHS`) bila perlu.

In [ ]:
# Model ringan: MLP Classifier (pendamping), Vanilla AE, Sparse AE, VAE
!python -m src.train --model mlp
!python -m src.train --model ae
!python -m src.train --model sparse_ae
!python -m src.train --model vae

In [ ]:
# GAN & DCGAN
!python -m src.train --model gan
!python -m src.train --model dcgan

In [ ]:
# DDPM (paling lama)
!python -m src.train --model ddpm

## 4. Eksperimen pembuktian batasan MLP Classifier (Tahap 1–5)
Menghasilkan `results/mlp_proof.json` dan `results/mlp_proof_tensors.pt` (bahan Figur 3).

In [ ]:
!python -m src.mlp_proof

## 5. Evaluasi kuantitatif (tabel Bagian IV)
MSE, PSNR, SSIM, FID, latency per batch 64, waktu training per epoch.
FID DDPM memakai sampling T=1000 langkah → bagian ini memakan beberapa menit.

In [ ]:
!python -m src.evaluate

## 6. Figur 1–5

In [ ]:
!python -m src.figures

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob(os.environ["DL_OUT_DIR"] + "/figures/*.png")):
    print(os.path.basename(f)); display(Image(f))

## 7. Lihat tabel metrik & hasil pembuktian

In [ ]:
import pandas as pd, json
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
display(pd.read_csv(os.environ["DL_OUT_DIR"] + "/results/metrics_table.csv"))
print(json.dumps(json.load(open(os.environ["DL_OUT_DIR"] + "/results/mlp_proof.json")), indent=1)[:3000])

## 8. (Opsional) Varian GAN loss minimax — untuk analisis vanishing gradient (Q3)
Hasil disimpan terpisah sebagai `gan_minimax` / `dcgan_minimax`; bandingkan kolom `G_grad_norm` di `results/history_*.json`.

In [ ]:
!python -m src.train --model gan --gan-loss minimax